Features:
- distance_from_home: continua
- distance_from_last_transaction: continua
- ratio_to_median_purchase_price: continua
- repeat_retailer: binaria
- used_chip: binaria
- used_pin_number: binaria
- online_order: binaria

Target:
- fraud: binaria

In [82]:
import pandas as pd
import numpy as np

In [83]:
# MANTER ORDEM DAS COLUNAS PARA EVITAR ERROS DE INDEXAÇÃO
COLUNAS_CONTINUAS = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price"
]

COLUNAS_BINARIAS = [
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

In [84]:
def gen_continuas(df_train_fraud, n_sintetico):
    """
    Gera dados sintéticos contínuos usando distribuição multivariada normal
    """
    X_cont = df_train_fraud[COLUNAS_CONTINUAS].values
    mu = X_cont.mean(axis=0)
    Sigma = np.cov(X_cont, rowvar=False)
    rng = np.random.default_rng(seed=4267)
    X_cont_sintetico = rng.multivariate_normal(mu, Sigma, size=n_sintetico)
    return X_cont_sintetico

def gen_binarias(df_train_fraud, n_sintetico):
    """
    Gera dados sintéticos binários usando Bernoulli

    """
    prob_binarias = {}
    rng = np.random.default_rng(seed=4267)

    for col in COLUNAS_BINARIAS:
        prob_binarias[col] = df_train_fraud[col].mean()
    
    binarias_sinteticas = {}
    for col in COLUNAS_BINARIAS:
        p = prob_binarias[col]
        binarias_sinteticas[col] = rng.binomial(1, p, size=n_sintetico)
    
    return binarias_sinteticas

In [85]:
df = pd.read_csv("dataset/card_transdata.csv")

In [86]:
# Separar train e teste para depois gerar os dados sintéticos apenas no conjunto de treino
from sklearn.model_selection import train_test_split
# Drop da coluna target
X = df[COLUNAS_CONTINUAS + COLUNAS_BINARIAS].values
y = df["fraud"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4267, stratify=y)

In [87]:
df_train = pd.DataFrame(X_train, columns=COLUNAS_CONTINUAS + COLUNAS_BINARIAS)
# SELECIONA APENAS LINHAS COM FRAUD = 1
fraud_data = df_train[y_train == 1.0]   

# GERAR QUANTIDADE NECESSARIA PARA BALANCEAR O DATASET
n_sintetico = int(abs((y_train == 1.0).sum() - (y_train == 0.0).sum()) // 1.5)
# GERAR DADOS (FEATURES CONTINUAS) PARA A CLASSE MINORITÁRIA (FRAUDULENTAS)
X_cont_sintetico = gen_continuas(fraud_data, n_sintetico)

# GERAR DADOS (FEATURES BINÁRIAS) PARA A CLASSE MINORITÁRIA (FRAUDULENTAS)
X_bin_sintetico = gen_binarias(fraud_data, n_sintetico)
df_sintetico = pd.DataFrame(X_cont_sintetico, columns=COLUNAS_CONTINUAS)

for col in COLUNAS_BINARIAS:
    df_sintetico[col] = X_bin_sintetico[col]

df_sintetico["fraud"] = 1.0

In [88]:
df_train["fraud"] = y_train

df_balanceado_train = pd.concat(
    [df_train, df_sintetico],
    ignore_index=True
)

print(df_balanceado_train["fraud"].value_counts())

fraud
0.0    730078
1.0    510026
Name: count, dtype: int64


In [89]:
X_train_balanceado = df_balanceado_train.drop(columns=["fraud"]).values
y_train_balanceado = df_balanceado_train["fraud"].values

In [90]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis        
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_balanceado, y_train_balanceado)
y_pred = lda.predict(X_test)

In [91]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {acc}")
print(f"Precision: {prec}")
print(f"Recall: {rec}")
print(f"F1-Score: {f1}")


Accuracy: 0.958935
Precision: 0.7358269720101781
Recall: 0.8271265945884103
F1-Score: 0.778810158627562


In [92]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis        

qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_balanceado, y_train_balanceado)
y_pred = qda.predict(X_test)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {acc}")
print(f"Precision: {prec}")
print(f"Recall: {rec}")
print(f"F1-Score: {f1}")


Accuracy: 0.942525
Precision: 0.6082224472085623
Recall: 0.9622447228419427
F1-Score: 0.7453308815384275
